# 042: Prompt sanity check — сравнение старого и balanced prompt

Цель ноутбука — перед большим end-to-end прогоном проверить, не провоцирует ли старый промпт слишком много отказов `I don't know`.

Ноутбук **не обращается к Qdrant** и **не пересобирает контексты**. Он берёт уже сохранённые controlled-ответы из `03a`, использует сохранённый `final_context`/`input_context` и заново генерирует ответ по двум промптам:

1. **old_strict** — старый строгий промпт: если информации недостаточно, отвечать `I don't know`.
2. **balanced_contextual** — новый сбалансированный промпт: если в контексте есть хоть какие-то релевантные зацепки, дать наиболее вероятный ответ, но **только по контексту**.

Проверка сделана маленькой: по умолчанию берётся `noise_level = 80`, три основных вида шума и несколько вопросов на каждый тип. Это нужно только для sanity check, а не для финального эксперимента.

In [49]:
# При необходимости установи зависимости один раз
!pip install groq tenacity pandas tqdm

In [50]:
import os
import re
import json
import time
import random
from pathlib import Path
from collections import defaultdict

import pandas as pd
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from groq import Groq, RateLimitError, APIError, BadRequestError

random.seed(42)

In [51]:
from pathlib import Path
import json

CONTROLLED_ANSWERS_DIR = Path("answers/controlled")

print("Exists:", CONTROLLED_ANSWERS_DIR.exists())
print("Path:", CONTROLLED_ANSWERS_DIR.resolve())

files = sorted(CONTROLLED_ANSWERS_DIR.rglob("*.json"))
print("JSON files:", len(files))

for p in files[:30]:
    print(p)

Exists: False
Path: /content/answers/controlled
JSON files: 0


In [52]:
for p in sorted(Path(".").rglob("*.json"))[:100]:
    print(p)


.config/.last_update_check.json
drive/MyDrive/Google AI Studio/applet_access_history.json
drive/MyDrive/Google AI Studio/package-lock.json
drive/MyDrive/Google AI Studio/package.json
drive/MyDrive/Google AI Studio/raw_text_edited.json
drive/MyDrive/rag_experiment/artifacts/answers/controlled/baseline__counterfactuals__lvl0.json
drive/MyDrive/rag_experiment/artifacts/answers/controlled/baseline__counterfactuals__lvl40.json
drive/MyDrive/rag_experiment/artifacts/answers/controlled/baseline__counterfactuals__lvl80.json
drive/MyDrive/rag_experiment/artifacts/answers/controlled/baseline__duplicates__lvl0.json
drive/MyDrive/rag_experiment/artifacts/answers/controlled/baseline__duplicates__lvl40.json
drive/MyDrive/rag_experiment/artifacts/answers/controlled/baseline__duplicates__lvl80.json
drive/MyDrive/rag_experiment/artifacts/answers/controlled/baseline__semantic_distractors__lvl0.json
drive/MyDrive/rag_experiment/artifacts/answers/controlled/baseline__semantic_distractors__lvl40.json
drive

In [53]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Настройки

Проверь пути. Если controlled-ответы лежат в другой папке, измени `CONTROLLED_ANSWERS_DIR`.

Рекомендуемый первый запуск:

- `TEST_METHODS = ["baseline"]`;
- `SAMPLE_PER_CONFIG = 5`;
- `NOISE_LEVELS = [80]`.

Это даст: `3 типа шума × 5 вопросов × 2 промпта = 30` генераций.

In [ ]:
# === API ===
GROQ_API_KEY = ' '
assert GROQ_API_KEY, "Не найден GROQ_API_KEY в переменных окружения"

groq_cli = Groq(api_key=GROQ_API_KEY)

# Модель можно поставить ту же, что в 03a/041
GENERATOR_MODEL = os.environ.get("GENERATOR_MODEL", "llama-3.3-70b-versatile")
GENERATOR_MAX_TOKENS = 80
GROQ_TPM_BUDGET = int(os.environ.get("GROQ_TPM_BUDGET", "5500"))

# === Пути ===
CONTROLLED_ANSWERS_DIR = Path("/content/drive/MyDrive/rag_experiment/artifacts/answers/controlled")
OUT_DIR = Path("prompt_tests")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# === Что тестируем ===
TEST_METHODS = ["baseline"]  # можно добавить "rerank", "hyde", "query_expansion"
NOISE_TYPES = ["semantic_distractors", "counterfactuals", "structural"]
NOISE_LEVELS = [80]
SAMPLE_PER_CONFIG = 5

# Если True — будет брать первые SAMPLE_PER_CONFIG вопросов, если False — случайную выборку
DETERMINISTIC_HEAD_SAMPLE = False

print("Model:", GENERATOR_MODEL)
print("Controlled answers dir:", CONTROLLED_ANSWERS_DIR.resolve())

Model: llama-3.3-70b-versatile
Controlled answers dir: /content/drive/MyDrive/rag_experiment/artifacts/answers/controlled


## 2. Защита от лимитов и обрывов

При ошибке лимитов ноутбук сохраняет кэш и останавливается. После восстановления лимитов можно просто перезапустить — уже готовые ответы не будут пересчитаны.

In [55]:
class ExperimentLimitReached(RuntimeError):
    pass

class GroqTokenBudget:
    def __init__(self, tpm_limit):
        self.tpm_limit = tpm_limit
        self.usage = []

    def _prune(self):
        cutoff = time.time() - 60
        self.usage = [(t, n) for t, n in self.usage if t > cutoff]

    def wait_if_needed(self, est_tokens):
        self._prune()
        used = sum(n for _, n in self.usage)
        if used + est_tokens > self.tpm_limit and self.usage:
            wait_sec = max(0, 60 - (time.time() - self.usage[0][0])) + 1
            print(f"⏳ TPM budget: sleep {wait_sec:.1f}s")
            time.sleep(wait_sec)
            self._prune()

    def record(self, tokens):
        self.usage.append((time.time(), tokens))

budget = GroqTokenBudget(GROQ_TPM_BUDGET)

def estimate_tokens(text):
    return max(1, len(text) // 4)

def is_groq_limit_error(exc):
    text = str(exc).lower()
    return (
        isinstance(exc, RateLimitError)
        or "rate limit" in text
        or "quota" in text
        or "tokens per minute" in text
        or "too many requests" in text
    )

@retry(
    stop=stop_after_attempt(2),
    wait=wait_exponential(multiplier=1, min=2, max=8),
    retry=retry_if_exception_type((APIError, BadRequestError)),
    reraise=True,
)
def groq_complete(prompt, model=GENERATOR_MODEL, max_tokens=80, temperature=0.0):
    est = estimate_tokens(prompt) + max_tokens
    budget.wait_if_needed(est)
    try:
        resp = groq_cli.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        text = resp.choices[0].message.content.strip()
        used = getattr(resp.usage, "total_tokens", None) or est
        budget.record(used)
        return text
    except Exception as e:
        if is_groq_limit_error(e):
            raise ExperimentLimitReached(str(e))
        raise

## 3. Старый и новый промпт

Новый промпт специально снижает лишние отказы, но сохраняет ключевое ограничение: **не использовать знания вне контекста**.

In [56]:
OLD_STRICT_PROMPT = """Answer the question based ONLY on the provided context.
Be concise — give the answer in 1-3 words when possible.
If the context does not contain enough information, respond with: "I don't know".

Context:
{context}

Question:
{question}

Answer:"""

BALANCED_CONTEXTUAL_PROMPT = """You are a question-answering assistant. Answer the question using ONLY the provided context.

The context may contain noisy, irrelevant, duplicated, corrupted, or contradictory fragments.
Your task is to identify the most relevant evidence and provide the most likely answer based only on that evidence.

Important rules:
1. Do not use outside knowledge.
2. Do not mention that the context is noisy unless it is necessary.
3. If the context contains at least some relevant evidence, give the best possible short answer.
4. If several fragments conflict, prefer the fragment that is most directly related to the question.
5. Answer exactly "I don't know" only if the provided context contains no relevant information for answering the question.
6. Keep the answer concise: usually 1-5 words, unless a short phrase is necessary.

Question:
{question}

Context:
{context}

Answer:"""

PROMPTS = {
    "old_strict": OLD_STRICT_PROMPT,
    "balanced_contextual": BALANCED_CONTEXTUAL_PROMPT,
}

## 4. Загрузка controlled-контекстов из 03a

Для честного сравнения промптов берём **один и тот же контекст** и меняем только инструкцию модели.

In [57]:
def find_config_file(method, noise_type, noise_level):
    candidates = [
        CONTROLLED_ANSWERS_DIR / f"{method}__{noise_type}__lvl{noise_level}.json",
        CONTROLLED_ANSWERS_DIR / f"{method}_{noise_type}_lvl{noise_level}.json",
        CONTROLLED_ANSWERS_DIR / method / f"{noise_type}__lvl{noise_level}.json",
        CONTROLLED_ANSWERS_DIR / method / f"{noise_type}_lvl{noise_level}.json",
    ]
    for p in candidates:
        if p.exists():
            return p
    patterns = [
        f"**/{method}*{noise_type}*lvl{noise_level}*.json",
        f"**/{method}*{noise_type}*{noise_level}*.json",
    ]
    for pat in patterns:
        found = sorted(CONTROLLED_ANSWERS_DIR.glob(pat))
        if found:
            return found[0]
    return None


def load_json_any(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        rows = []
        for qid, rec in data.items():
            if isinstance(rec, dict):
                r = dict(rec)
                r.setdefault("qid", qid)
                rows.append(r)
        return rows
    if isinstance(data, list):
        return data
    raise ValueError(f"Неожиданный формат JSON: {path}")


def get_context_docs(rec):
    for key in ["final_context", "input_context", "context_docs", "context"]:
        val = rec.get(key)
        if isinstance(val, list) and val:
            return val
    return []


def get_answer_text(rec):
    for key in ["answer", "predicted", "prediction", "generated_answer"]:
        val = rec.get(key)
        if isinstance(val, str):
            return val
    return ""


def get_gold_text(rec):
    for key in ["gold", "gold_answer", "answer_gold", "reference", "expected"]:
        val = rec.get(key)
        if isinstance(val, str):
            return val
        if isinstance(val, list) and val:
            return str(val[0])
    return ""


def get_question_text(rec):
    for key in ["question", "query"]:
        val = rec.get(key)
        if isinstance(val, str):
            return val
    return ""


def format_context(docs, max_chars_per_doc=1200):
    parts = []
    for i, d in enumerate(docs, 1):
        if isinstance(d, str):
            title, role, text = "", "", d
        else:
            title = d.get("title", "")
            role = d.get("role", "")
            text = d.get("text", "") or d.get("content", "") or ""
        text = text[:max_chars_per_doc]
        header = f"[{i}]"
        if title:
            header += f" Title: {title}"
        if role:
            header += f" | Role: {role}"
        parts.append(f"{header}\n{text}")
        return "\n\n".join(parts)

In [58]:
def collect_prompt_test_items():
    items = []
    missing = []
    for method in TEST_METHODS:
        for noise_type in NOISE_TYPES:
            for lvl in NOISE_LEVELS:
                p = find_config_file(method, noise_type, lvl)
                if not p:
                    missing.append((method, noise_type, lvl))
                    continue
                rows = load_json_any(p)
                usable = []
                for rec in rows:
                    q = get_question_text(rec)
                    docs = get_context_docs(rec)
                    if q and docs:
                        usable.append(rec)
                if not usable:
                    continue
                if DETERMINISTIC_HEAD_SAMPLE:
                    sample = usable[:SAMPLE_PER_CONFIG]
                else:
                    sample = random.sample(usable, min(SAMPLE_PER_CONFIG, len(usable)))
                for rec in sample:
                    qid = rec.get("qid", rec.get("id", ""))
                    items.append({
                        "qid": qid,
                        "method": method,
                        "noise_type": noise_type,
                        "noise_level": lvl,
                        "question": get_question_text(rec),
                        "gold": get_gold_text(rec),
                        "old_answer_from_03a": get_answer_text(rec),
                        "context_docs": get_context_docs(rec),
                        "source_file": str(p),
                    })
    return items, missing

items, missing = collect_prompt_test_items()
print("Items to test:", len(items))
if missing:
    print("Missing configs:")
    for x in missing:
        print(" -", x)

pd.DataFrame([{k:v for k,v in it.items() if k != "context_docs"} for it in items]).head()

Items to test: 15


,qid,method,noise_type,noise_level,question,gold,old_answer_from_03a,source_file
0,5ae1bf46554299234fd042e6,baseline,semantic_distractors,80,Are both Jack and Coke and Clover Club Cocktai...,yes,I don't know,/content/drive/MyDrive/rag_experiment/artifact...
1,5ab802085542990e739ec7ca,baseline,semantic_distractors,80,What is the uppermost age range for the sort o...,early 20s,Young Adult,/content/drive/MyDrive/rag_experiment/artifact...
2,5a8725a15542996432c57230,baseline,semantic_distractors,80,What other method of extending an ice hockey g...,the shootout,Shootout,/content/drive/MyDrive/rag_experiment/artifact...
3,5ab6e865554299710c8d1fad,baseline,semantic_distractors,80,Between Danny Elfman and Fran Healy who has wo...,Elfman,Danny Elfman,/content/drive/MyDrive/rag_experiment/artifact...
4,5ae8202255429952e35eaa3a,baseline,semantic_distractors,80,What Japanese fashion label was founded by the...,Comme des Garçons,Comme des Garçons,/content/drive/MyDrive/rag_experiment/artifact...


## 5. Метрики для быстрого сравнения

Здесь считаются дешёвые метрики:

- `EM`;
- `F1`;
- `is_idk`.

Этого достаточно для первого решения: менять промпт или нет.

In [59]:
IDK_PATTERNS = [
    r"^\s*i\s+don['’]?t\s+know\s*\.?\s*$",
    r"^\s*unknown\s*\.?\s*$",
    r"^\s*not\s+enough\s+information\s*\.?\s*$",
    r"^\s*cannot\s+answer\s*\.?\s*$",
    r"^\s*can't\s+answer\s*\.?\s*$",
]

def is_idk(text):
    t = (text or "").strip().lower()
    return any(re.search(p, t) for p in IDK_PATTERNS)


def normalize_answer(s):
    s = (s or "").lower().strip()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = " ".join(s.split())
    return s


def exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))


def f1_score(pred, gold):
    pred_toks = normalize_answer(pred).split()
    gold_toks = normalize_answer(gold).split()
    if len(pred_toks) == 0 and len(gold_toks) == 0:
        return 1.0
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    common = defaultdict(int)
    for t in gold_toks:
        common[t] += 1
    num_same = 0
    for t in pred_toks:
        if common[t] > 0:
            num_same += 1
            common[t] -= 1
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

## 6. Запуск сравнения промптов

Результаты сохраняются в кэш `prompt_tests/prompt_compare_cache.json`.

In [60]:
CACHE_PATH = OUT_DIR / "prompt_compare_cache.json"

if CACHE_PATH.exists():
    with open(CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
else:
    cache = {}

def save_cache():
    tmp = CACHE_PATH.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)
    tmp.replace(CACHE_PATH)


def make_cache_key(item, prompt_name):
    return "::".join([
        str(item["method"]),
        str(item["noise_type"]),
        str(item["noise_level"]),
        str(item["qid"]),
        prompt_name,
    ])


def run_one_prompt(item, prompt_name, prompt_template):
    context = format_context(item["context_docs"])
    prompt = prompt_template.format(question=item["question"], context=context)
    return groq_complete(prompt, max_tokens=GENERATOR_MAX_TOKENS, temperature=0.0)

try:
    for item in tqdm(items, desc="Prompt test items"):
        for prompt_name, prompt_template in PROMPTS.items():
            key = make_cache_key(item, prompt_name)
            if key in cache:
                continue
            answer = run_one_prompt(item, prompt_name, prompt_template)
            cache[key] = {
                "qid": item["qid"],
                "method": item["method"],
                "noise_type": item["noise_type"],
                "noise_level": item["noise_level"],
                "prompt_name": prompt_name,
                "question": item["question"],
                "gold": item["gold"],
                "answer": answer,
                "old_answer_from_03a": item["old_answer_from_03a"],
                "source_file": item["source_file"],
            }
            save_cache()
except ExperimentLimitReached as e:
    save_cache()
    print("⛔ Groq limit/quota reached. Кэш сохранён, перезапусти позже.")
    print(str(e)[:500])

print("Cached generations:", len(cache))

Prompt test items:   0%|          | 0/15 [00:00<?, ?it/s]

⏳ TPM budget: sleep 56.2s
⏳ TPM budget: sleep 1.2s
Cached generations: 30


## 7. Сводка: какой промпт даёт меньше `I don't know`

Главная таблица — по `prompt_name × noise_type × noise_level`.

In [61]:
rows = list(cache.values())
res = pd.DataFrame(rows)
if len(res) == 0:
    print("Пока нет результатов. Запусти предыдущую ячейку.")
else:
    res["is_idk"] = res["answer"].apply(is_idk)
    res["EM"] = res.apply(lambda r: exact_match(r["answer"], r["gold"]), axis=1)
    res["F1"] = res.apply(lambda r: f1_score(r["answer"], r["gold"]), axis=1)
    summary = (
        res.groupby(["prompt_name", "method", "noise_type", "noise_level"])
        .agg(
            n=("qid", "count"),
            idk_count=("is_idk", "sum"),
            idk_rate=("is_idk", "mean"),
            EM=("EM", "mean"),
            F1=("F1", "mean"),
        )
        .reset_index()
        .sort_values(["method", "noise_type", "noise_level", "prompt_name"])
    )
    display(summary)
    res.to_csv(OUT_DIR / "prompt_compare_detailed.csv", index=False)
    summary.to_csv(OUT_DIR / "prompt_compare_summary.csv", index=False)
    print("Saved:", OUT_DIR / "prompt_compare_detailed.csv")
    print("Saved:", OUT_DIR / "prompt_compare_summary.csv")

,prompt_name,method,noise_type,noise_level,n,idk_count,idk_rate,EM,F1
0,balanced_contextual,baseline,counterfactuals,80,5,0,0.0,0.2,0.644058
3,old_strict,baseline,counterfactuals,80,5,3,0.6,0.0,0.293333
1,balanced_contextual,baseline,semantic_distractors,80,5,0,0.0,0.2,0.433333
4,old_strict,baseline,semantic_distractors,80,5,3,0.6,0.2,0.200000
2,balanced_contextual,baseline,structural,80,5,1,0.2,0.0,0.133333
5,old_strict,baseline,structural,80,5,4,0.8,0.0,0.000000


Saved: prompt_tests/prompt_compare_detailed.csv
Saved: prompt_tests/prompt_compare_summary.csv


## 8. Парные сравнения ответов

Эта таблица нужна для ручной проверки: старый и новый промпт рядом на одном и том же вопросе и контексте.

In [62]:
if len(cache) > 0:
    res = pd.DataFrame(list(cache.values()))
    pivot = res.pivot_table(
        index=["method", "noise_type", "noise_level", "qid", "question", "gold"],
        columns="prompt_name",
        values="answer",
        aggfunc="first"
    ).reset_index()
    if "old_strict" in pivot.columns:
        pivot["old_is_idk"] = pivot["old_strict"].apply(is_idk)
    if "balanced_contextual" in pivot.columns:
        pivot["balanced_is_idk"] = pivot["balanced_contextual"].apply(is_idk)
    pivot.to_csv(OUT_DIR / "prompt_compare_pairs.csv", index=False)
    display(pivot.head(20))
    print("Saved:", OUT_DIR / "prompt_compare_pairs.csv")

prompt_name,method,noise_type,noise_level,qid,question,gold,balanced_contextual,old_strict,old_is_idk,balanced_is_idk
0,baseline,counterfactuals,80,5ab6e865554299710c8d1fad,Between Danny Elfman and Fran Healy who has wo...,Elfman,Danny Elfman.,Danny Elfman,False,False
1,baseline,counterfactuals,80,5adc983c5542990d50227cb6,In what Country is Sul America Esporte Clube in?,Brazil,Brazil,I don't know,True,False
2,baseline,counterfactuals,80,5ade15f35542997545bbbe47,Katherine Waterston and Chrisann Brennan has w...,Steve Jobs,Steve Jobs connection,I don't know,True,False
3,baseline,counterfactuals,80,5ae5aba0554299546bf82f17,Which of Tara Strong major voice role in anima...,Teen Titans Go!,"Raven (not mentioned), but likely ""Teen Titans...",Teen Titans,False,False
4,baseline,counterfactuals,80,5ae736cb5542991bbc9761c2,This aircraft carrier served as a recovery shi...,three,Three times,I don't know,True,False
5,baseline,semantic_distractors,80,5a8725a15542996432c57230,What other method of extending an ice hockey g...,the shootout,Overtime.,Shootout,False,False
6,baseline,semantic_distractors,80,5ab6e865554299710c8d1fad,Between Danny Elfman and Fran Healy who has wo...,Elfman,Danny Elfman.,I don't know,True,False
7,baseline,semantic_distractors,80,5ab802085542990e739ec7ca,What is the uppermost age range for the sort o...,early 20s,Young Adult.,Young Adult,False,False
8,baseline,semantic_distractors,80,5ae1bf46554299234fd042e6,Are both Jack and Coke and Clover Club Cocktai...,yes,"Yes, both are.",I don't know,True,False
9,baseline,semantic_distractors,80,5ae8202255429952e35eaa3a,What Japanese fashion label was founded by the...,Comme des Garçons,Comme des Garçons.,I don't know,True,False


Saved: prompt_tests/prompt_compare_pairs.csv


## 9. Как принять решение

Рекомендуемая логика:

1. Если `balanced_contextual` заметно снижает `idk_rate`, но EM/F1 не падают резко — можно заменить основной промпт в большом end-to-end ноутбуке.
2. Если `balanced_contextual` снижает `I don't know`, но начинает часто выбирать ложный counterfactual, это будет видно по ручной таблице. Тогда лучше оставить строгий или сделать промежуточную версию.
3. Для финального диплома важно зафиксировать, что промпт не заставляет модель отвечать из внешних знаний, а только уменьшает избыточные отказы при наличии релевантных фрагментов в контексте.